## 2. Geocoding – Koordinaten für Start- und Zielort

Für die spätere Wetterabfrage über eine externe API werden geografische Koordinaten
(Breitengrad & Längengrad) für jede Etappe benötigt. Da der Datensatz bereits die
Ortsnamen von Start (`departure`) und Ziel (`arrival`) jeder Etappe enthält, werden
diese über **Reverse Geocoding** in konkrete Koordinaten überführt.

Als Geocoding-Dienst kommt [**Nominatim**](https://nominatim.org/) der OpenStreetMap-Community
zum Einsatz – ein kostenfreier Dienst, der keinen API-Schlüssel benötigt.

**Arbeitsschritte in diesem Notebook:**

1. **Pickle laden** – Einlesen des bereinigten Master-Datensatzes aus dem vorherigen Checkpoint.
2. **Geocoding einrichten** – Initialisierung von Nominatim mit einem `RateLimiter`, um die
   API-Nutzungsregeln einzuhalten (min. 1 Sekunde zwischen Anfragen).
3. **Koordinaten ermitteln** – Alle einzigartigen Ortsnamen aus `departure` und `arrival`
   werden gesammelt und einmalig geocodiert. Ein lokaler Cache (`coords_cache`) vermeidet
   redundante API-Aufrufe für denselben Ort.
4. **Spalten befüllen** – Die ermittelten Koordinaten werden in vier neue Spalten eingetragen:
   `departure_lat`, `departure_lon`, `arrival_lat`, `arrival_lon`.
5. **Speichern** – Der erweiterte Datensatz wird als neues Pickle-File persistiert.

> ⚠️ **Hinweis:** Nominatim ist ein öffentlicher Dienst und darf laut Nutzungsbedingungen
> nur mit einer Anfrage pro Sekunde genutzt werden. Der `RateLimiter` stellt dies sicher.
> Bei sehr großen Datensätzen kann dieser Schritt entsprechend Zeit in Anspruch nehmen.

---

In [ ]:
import pickle
import requests
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# --- 1. Pickle laden ---
with open('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/01_cleaned_master_data.pkl', "rb") as f:
    df = pickle.load(f)

# --- 2. Geocoding einrichten ---
geolocator = Nominatim(user_agent="mein_wetter_projekt")            
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

def get_coords(ortsname):
    try:
        loc = geocode(str(ortsname))
        if loc:
            return loc.latitude, loc.longitude
    except Exception:
        pass
    return None, None

# --- 3. Koordinaten für departure und arrival ermitteln ---
# Alle einzigartigen Orte sammeln (spart API-Aufrufe!)
alle_orte = pd.unique(df[["departure", "arrival"]].values.ravel())

coords_cache = {}
for ort in alle_orte:
    if pd.notna(ort):
        coords_cache[ort] = get_coords(ort)
        print(f"{ort}: {coords_cache[ort]}")

# --- 4. Koordinaten in neue Spalten eintragen ---
df["departure_lat"] = df["departure"].map(lambda o: coords_cache.get(o, (None, None))[0])
df["departure_lon"] = df["departure"].map(lambda o: coords_cache.get(o, (None, None))[1])
df["arrival_lat"]   = df["arrival"].map(lambda o: coords_cache.get(o, (None, None))[0])
df["arrival_lon"]   = df["arrival"].map(lambda o: coords_cache.get(o, (None, None))[1])

# --- 5. Speichern ---
df.to_pickle("cleaned_master_data_with_coordinates.pkl")
#df.to_csv("cleaned_master_data_with_coordinates.csv", index=False)

print(df[["departure", "departure_lat", "departure_lon",
          "arrival",   "arrival_lat",   "arrival_lon"]].head())

📍 Fromentine: (46.8896706, -2.1366623)
📍 Noirmoutier-en-l'Ile: (47.000089, -2.2441449)


KeyboardInterrupt: 

## 3. Wetterdaten-Anreicherung via Open-Meteo API

Um den Einfluss von Witterungsbedingungen auf das Rennergebnis modellieren zu können,
wird der Datensatz um historische Wetterdaten erweitert. Für jede Etappe werden sowohl
am **Startort** (`departure`) als auch am **Zielort** (`arrival`) die meteorologischen
Bedingungen zum jeweiligen Renntag abgefragt.

Als Datenquelle dient die kostenfreie [**Open-Meteo Archive API**](https://open-meteo.com/),
die stundengenaue Wetterwerte für beliebige Koordinaten und historische Datumsangaben liefert –
ohne API-Schlüssel.

**Abgefragte Wettervariablen:**

| Variable (API)          | Zielspalte                   | Beschreibung                         |
|-------------------------|------------------------------|--------------------------------------|
| `temperature_2m`        | `*_temp_mittel`              | Temperatur in 2 m Höhe (°C)          |
| `rain`                  | `*_regen_mittel`             | Regen (mm)                           |
| `wind_speed_10m`        | `*_wind_mittel`              | Windgeschwindigkeit in 10 m (km/h)   |
| `relativehumidity_2m`   | `*_luftfeuchte_mittel`       | Relative Luftfeuchtigkeit (%)        |
| `precipitation`         | `*_niederschlag_mittel`      | Gesamtniederschlag (mm)              |
| `winddirection_10m`     | `*_windrichtung_mittel`      | Windrichtung in 10 m (°)             |

**Arbeitsschritte in diesem Notebook:**

1. **Laden** – Einlesen des Datensatzes mit Koordinaten aus dem vorherigen Checkpoint.
2. **Wetterdaten-Funktion** – Eine Hilfsfunktion ruft die stündlichen Wetterdaten für
   eine gegebene Koordinate und ein Datum ab. Bei einem Rate-Limit-Fehler (HTTP 429)
   wird automatisch 60 Minuten gewartet und der Aufruf wiederholt.
3. **Mittelwert-Funktion** – Die stündlichen Rohdaten werden auf einen relevanten
   Zeitfenster-Mittelwert aggregiert: `12–15 Uhr` für den Startort, `15–18 Uhr` für
   den Zielort – als grobe Annäherung an die tatsächliche Rennzeit.
4. **Unique-Kombinations-Optimierung** – Um die Anzahl der API-Aufrufe zu minimieren,
   werden ausschließlich einzigartige `(Koordinate, Datum)`-Kombinationen abgefragt.
   Die Ergebnisse werden anschließend per Left-Join zurück in den Haupt-DataFrame
   gespielt.
5. **Rate-Limiting** – Zwischen jeder Anfrage wird 3 Sekunden gewartet, um das
   API-Limit von max. 20 Anfragen/Minute einzuhalten.
6. **Speichern** – Der vollständig angereicherte Datensatz wird als neues Pickle-File
   persistiert.

> ⚠️ **Hinweis:** Aufgrund des 3-Sekunden-Delays und der Vielzahl einzigartiger
> Etappen-Koordinaten kann dieser Schritt mehrere Stunden in Anspruch nehmen.
> Es empfiehlt sich, den Code unbeaufsichtigt im Hintergrund laufen zu lassen.

In [ ]:
import pickle
import requests
import pandas as pd
import time

# --- 1. Laden ---
with open("/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/cleaned_master_data_with_coordinates.pkl", "rb") as f:
    df = pickle.load(f)

df["date"] = pd.to_datetime(df["date"])

# --- 2. Wetterdaten-Funktion ---
def get_wetter(lat, lon, datum):
    if pd.isna(lat) or pd.isna(lon):
        return {}
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}&longitude={lon}"
        f"&start_date={datum}&end_date={datum}"
        f"&hourly=temperature_2m,rain,wind_speed_10m,"
        f"relativehumidity_2m,precipitation,winddirection_10m"
        f"&timezone=Europe%2FBerlin"
    )
    response = requests.get(url)
    if response.status_code == 200:
        return response.json().get("hourly", {})
    elif response.status_code == 429:
        print("⚠️ Rate Limit – 3600s warten...")
        time.sleep(3600)
        response2 = requests.get(url)
        if response2.status_code == 200:
            return response2.json().get("hourly", {})
    return {}

# --- 3. Hilfsfunktion: Mittelwert für Stundenbereich ---
def mittelwert_stundenbereich(daten, datum, stunden_von, stunden_bis, key):
    zeiten = daten.get("time", [])
    werte  = daten.get(key, [])
    gefiltert = [
        wert for zeit, wert in zip(zeiten, werte)
        if zeit.startswith(datum)
        and stunden_von <= int(zeit[11:13]) <= stunden_bis
        and wert is not None
    ]
    return round(sum(gefiltert) / len(gefiltert), 2) if gefiltert else None

# --- 4. Stundenbereiche und Variablen ---
stunden_config = {
    "departure": (12, 15),
    "arrival":   (15, 18),
}

wetter_variablen = {
    "temperature_2m":      "temp_mittel",
    "rain":                "regen_mittel",
    "wind_speed_10m":      "wind_mittel",
    "relativehumidity_2m": "luftfeuchte_mittel",
    "precipitation":       "niederschlag_mittel",
    "winddirection_10m":   "windrichtung_mittel",
}

# --- 5. Nur unique Kombinationen abfragen und mergen ---
for prefix, (std_von, std_bis) in stunden_config.items():
    lat_col = f"{prefix}_lat"
    lon_col = f"{prefix}_lon"

    # Einzigartige Ort+Tag-Kombinationen
    unique = (
        df[[lat_col, lon_col, "date"]]
        .drop_duplicates()
        .dropna()
        .copy()
    )
    unique["datum_str"] = unique["date"].dt.strftime("%Y-%m-%d")

    print(f"{prefix}: {len(unique)} einzigartige API-Anfragen")

    # Wetterdaten nur für unique Kombinationen abrufen
    for api_key, spalten_suffix in wetter_variablen.items():
        unique[f"{prefix}_{spalten_suffix}"] = None

    for i, (idx, row) in enumerate(unique.iterrows()):
        if i % 100 == 0:
            print(f"⏳ {prefix}: {i}/{len(unique)}")

        time.sleep(3)  # 3 Sekunden Pause → max 20 Anfragen/Minute
        daten = get_wetter(row[lat_col], row[lon_col], row["datum_str"])

        for api_key, spalten_suffix in wetter_variablen.items():
            unique.at[idx, f"{prefix}_{spalten_suffix}"] = mittelwert_stundenbereich(
                daten, row["datum_str"], std_von, std_bis, api_key
            )

    # Ergebnisse per Merge zurück in den Haupt-DataFrame
    df = df.merge(
        unique[[lat_col, lon_col, "date"] + [f"{prefix}_{s}" for s in wetter_variablen.values()]],
        on=[lat_col, lon_col, "date"],
        how="left"
    )

    print(f"{prefix} fertig!")

# --- 6. Speichern ---
df.to_pickle("/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/cleaned_master_data_with_coordinates_and_weather.pkl")
print(df[[
    "departure", "departure_temp_mittel", "departure_regen_mittel",
    "departure_wind_mittel", "departure_luftfeuchte_mittel",
    "departure_niederschlag_mittel", "departure_windrichtung_mittel",
    "arrival", "arrival_temp_mittel", "arrival_regen_mittel",
    "arrival_wind_mittel", "arrival_luftfeuchte_mittel",
    "arrival_niederschlag_mittel", "arrival_windrichtung_mittel"
]].head())